# Visualização dos Resultados — Rotas Alternativas das OAEs do Ceará

Três figuras para artigo:
1. **Mapa de pontos** — OAEs coloridas pelo comprimento da rota alternativa
2. **Histograma** — distribuição das distâncias A1 e A2
3. **Scatter A1 × A2** — comparação entre as duas análises por OAE

Dependências: `pandas`, `geopandas`, `matplotlib`, `geobr`, `simplekml`, `scipy`  
Instalar: `pip install geobr geopandas matplotlib pandas openpyxl simplekml scipy`

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import numpy as np
from matplotlib.cm import ScalarMappable
from scipy import stats
centimeters=1/2.54
DPI=600
# Arquivos de entrada
CONSOLIDADO_CSV = 'consolidado.csv'
PLANILHA_OAES   = 'OAEs.xlsx'

# Resolução das figuras exportadas
DPI = 300

# Paleta de cores para as figuras
COR_A1 = '#2166ac'   # azul
COR_A2 = '#d6604d'   # vermelho-alaranjado
COR_SEM_ROTA = '#111111'

## 1. Carga e preparação dos dados

In [ ]:
# Consolidado
df = pd.read_csv(CONSOLIDADO_CSV, index_col='CodPro')
print(f'OAEs no consolidado: {len(df)}')
print(df[['status_A1','status_A2']].value_counts().head(20))

In [ ]:
# Planilha de OAEs (coordenadas)
# SGE não é usado aqui como chave: já vem em df (consolidado.csv), como coluna
# informativa (NaN para OAEs sem SGE atribuído). A chave de junção é CodPro.
df_oaes = pd.read_excel(PLANILHA_OAES, header=0)
col_rename = ['CodPro', 'SGE', 'Nome', 'Latitude', 'Longitude']
df_oaes.columns = col_rename + [f'extra_{i}' for i in range(len(df_oaes.columns) - 5)]
df_oaes = df_oaes.dropna(subset=['CodPro']).copy()
df_oaes = df_oaes.set_index('CodPro')[['Nome', 'Latitude', 'Longitude']]
print(f'OAEs na planilha: {len(df_oaes)}')

In [ ]:
# Join: consolidado + coordenadas
df_full = df.join(df_oaes, how='left')
df_full = df_full.dropna(subset=['Latitude', 'Longitude'])

# GeoDataFrame
gdf = gpd.GeoDataFrame(
    df_full,
    geometry=gpd.points_from_xy(df_full['Longitude'], df_full['Latitude']),
    crs=4326,
)

# Colunas auxiliares
gdf['encontrada_A1'] = gdf['status_A1'] == 'encontrada'
gdf['encontrada_A2'] = gdf['status_A2'] == 'encontrada'

print(f'OAEs com coordenadas: {len(gdf)}')
print(f'  A1 encontrada: {gdf["encontrada_A1"].sum()}  |  A2 encontrada: {gdf["encontrada_A2"].sum()}')

In [ ]:
# Limite do estado do Ceará via geobr
try:
    import geobr
    ceara = geobr.read_state(code_state='CE', year=2020)
    ceara = ceara.to_crs(4326)
    print('Limite do Ceará carregado via geobr.')
except Exception as e:
    print(f'geobr não disponível ({e}). Usando bounding box das OAEs.')
    ceara = None

### Vias federais e estaduais para o mapa

In [ ]:
import os
from pyproj import Transformer

GPKG_SNV    = 'SNV_ESTADUAL_planarizado.gpkg'
CRS_GPKG    = 5880   # SIRGAS 2000 / Brazil Polyconic (CRS do arquivo)

rodovias_fed = None
rodovias_est = None

# ── Bbox do Ceará em WGS84 ───────────────────────────────────
if ceara is not None:
    x0, y0, x1, y1 = ceara.total_bounds   # graus
else:
    x0, y0, x1, y1 = -42.0, -8.2, -37.2, -2.4

# Converte bbox para o CRS do GPKG (EPSG:5880, metros)
_tr = Transformer.from_crs(4326, CRS_GPKG, always_xy=True)
x0m, y0m = _tr.transform(x0, y0)
x1m, y1m = _tr.transform(x1, y1)
bbox_5880 = (x0m, y0m, x1m, y1m)

print(f'Bbox WGS84  : ({x0:.4f}, {y0:.4f}, {x1:.4f}, {y1:.4f})')
print(f'Bbox EPSG:5880: ({x0m:.0f}, {y0m:.0f}, {x1m:.0f}, {y1m:.0f})')
print(f'Arquivo existe: {os.path.exists(GPKG_SNV)}')

if not os.path.exists(GPKG_SNV):
    print(f'\nARQUIVO NÃO ENCONTRADO em {os.getcwd()}')
else:
    print(f'\nCarregando {GPKG_SNV}...')
    _rede = gpd.read_file(GPKG_SNV, bbox=bbox_5880)
    print(f'  Segmentos lidos  : {len(_rede):,}')
    print(f'  Valores _fonte   : {_rede["_fonte"].value_counts().to_dict()}')

    _rede = _rede.to_crs(4326)
    rodovias_fed = _rede[_rede['_fonte'] == 'SNV'].copy()
    rodovias_est = _rede[_rede['_fonte'] == 'ESTADUAL'].copy()

print(f'\nFederais prontas : {len(rodovias_fed) if rodovias_fed is not None else "None"}')
print(f'Estaduais prontas: {len(rodovias_est) if rodovias_est is not None else "None"}')

## Mapa da área de estudo — universo de OAEs e rede rodoviária federal

Mapa de contexto, no mesmo estilo gráfico da Figura 1: localização de todas as OAEs em análise e identificação das rodovias federais (SNV) que cortam o estado, com o nome de cada via anotado diretamente sobre o traçado correspondente.

In [ ]:
from shapely.ops import linemerge, unary_union
from matplotlib import patheffects as pe


def _ponto_e_angulo_rotulo(geom, passo=0.02):
    """Ponto no meio da via e ângulo (graus) para orientar o rótulo ao longo do traçado."""
    linha = linemerge(geom) if geom.geom_type == 'MultiLineString' else geom
    if linha.geom_type == 'MultiLineString':
        linha = max(linha.geoms, key=lambda g: g.length)

    p_meio = linha.interpolate(0.5, normalized=True)
    p_a = linha.interpolate(max(0.5 - passo, 0.0), normalized=True)
    p_b = linha.interpolate(min(0.5 + passo, 1.0), normalized=True)

    ang = np.degrees(np.arctan2(p_b.y - p_a.y, p_b.x - p_a.x))
    if ang > 90:
        ang -= 180
    elif ang < -90:
        ang += 180
    return p_meio, ang


# Recorta as vias federais aos limites do Ceará e agrupa por número da rodovia
# (uma mesma BR pode ter trechos cadastrados com sufixo de UF vizinha, ex. 'BR-116/PB',
# que ao serem recortados pelo polígono do Ceará representam o mesmo eixo rodoviário)
rotulos_vias_federais = []
if rodovias_fed is not None and not rodovias_fed.empty and ceara is not None:
    fed_ce = gpd.clip(rodovias_fed, ceara).copy()
    fed_ce['rodovia'] = fed_ce['_id_rodovia'].str.split('/').str[0]

    for rodovia, grupo in fed_ce.groupby('rodovia'):
        geom_uniao = unary_union(grupo.geometry.values)
        if geom_uniao.length == 0:
            continue
        p_meio, ang = _ponto_e_angulo_rotulo(geom_uniao)
        rotulos_vias_federais.append((rodovia, p_meio, ang))

print(f'Rótulos de vias federais: {len(rotulos_vias_federais)}')
print(sorted(r[0] for r in rotulos_vias_federais))

In [ ]:
fig, ax = plt.subplots(figsize=(12*centimeters, 12*centimeters), dpi=DPI)
ax.set_aspect('equal')

# ── Principais cidades do Ceará ────────────────────────────
cidades = [
    ('Fortaleza', -38.5267, -3.7319),
    ('Sobral', -40.3477, -3.6892),
    ('Juazeiro do Norte', -39.3156, -7.2130),
    ('Crato', -39.4103, -7.2318),
    ('Iguatu', -39.2987, -6.3597),
    ('Quixadá', -39.0150, -4.9712),
    ('Russas', -37.9693, -4.9400),
    ('Itapipoca', -39.5840, -3.4940),
    ('Limoeiro do Norte', -38.1000, -5.1450),
    ('Aracati', -37.7650, -4.5619),
    ('Camocim', -40.8411, -2.9000),
    ('Crateús', -40.6667, -5.1789),
]

for nome, lon, lat in cidades:
    ax.scatter(lon, lat, s=14, color='black', zorder=8)
    ax.text(
        lon + 0.03, lat + 0.03, nome,
        fontsize=6.5, color='black', zorder=9,
        path_effects=[pe.withStroke(linewidth=2.5, foreground='white', alpha=0.8)]
    )

# ── Fundo do estado ──────────────────────────────────────
if ceara is not None:
    ceara.plot(ax=ax, facecolor="#adadad", edgecolor='#555555', linewidth=0.8, zorder=1)
else:
    x0, y0, x1, y1 = gdf.total_bounds
    ax.set_xlim(x0 - 0.2, x1 + 0.2)
    ax.set_ylim(y0 - 0.2, y1 + 0.2)
    rect = plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                          facecolor="#9E9E9E", edgecolor='#555555', linewidth=0.8)
    ax.add_patch(rect)

# ── Vias estaduais ────────────────────────────────────────
n_est = 0
if rodovias_est is not None and not rodovias_est.empty:
    rodovias_est.plot(ax=ax, color="#e7e6e6", linewidth=0.5, zorder=2, alpha=0.7)
    n_est = len(rodovias_est)

# ── Vias federais ─────────────────────────────────────────
n_fed = 0
if rodovias_fed is not None and not rodovias_fed.empty:
    rodovias_fed.plot(ax=ax, color="#45CAFF", linewidth=1.1, zorder=3, alpha=0.9)
    n_fed = len(rodovias_fed)

# ── Nome das vias federais, sobre o próprio traçado ───────
for rodovia, ponto, ang in rotulos_vias_federais:
    ax.text(
        ponto.x, ponto.y, rodovia,
        fontsize=5, fontweight='bold', color='#0a4a66',
        rotation=ang, rotation_mode='anchor', ha='center', va='center', zorder=7,
        path_effects=[pe.withStroke(linewidth=2.2, foreground='white', alpha=0.3)],
    )

# ── OAEs em análise (universo completo) ───────────────────
ax.scatter(
    gdf.geometry.x, gdf.geometry.y,
    c='#c0392b', s=16, zorder=6, edgecolors='white', linewidths=0.3, alpha=0.9,
)

print(f'  federais={n_fed}  estaduais={n_est}  rótulos={len(rotulos_vias_federais)}  OAEs={len(gdf)}')

# ── Legenda ───────────────────────────────────────────────
handles = []
if n_fed:
    handles.append(plt.Line2D([0], [0], color='#45CAFF', linewidth=1.5, label='Via federal (SNV)'))
if n_est:
    handles.append(plt.Line2D([0], [0], color='#aaaaaa', linewidth=1.0, label='Via estadual'))
handles.append(plt.Line2D(
    [0], [0], marker='o', linestyle='',
    markerfacecolor='#c0392b', markeredgecolor='white',
    markersize=7, label=f'OAE em análise (n={len(gdf)})',
))
ax.legend(handles=handles, loc='lower left', fontsize=7.5, framealpha=0.88)

# ax.set_title('Área de estudo — OAEs do Ceará e rede rodoviária federal (SNV)',
            #  fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel('Longitude', fontsize=8)
ax.set_ylabel('Latitude', fontsize=8)
ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('mapa_area_estudo.png', dpi=DPI, bbox_inches='tight')
print('Salvo: mapa_area_estudo.png')
plt.show()

## Figura 1 — Mapa de pontos (rota alternativa A1 e A2)

In [ ]:
def _mapa_pontos(gdf_plot, analise, cor_mapa, titulo, nome_arquivo,
                 gdf_fed=None, gdf_est=None):
    col_km  = f'km_{analise}'
    col_enc = f'encontrada_{analise}'

    com_rota = gdf_plot[gdf_plot[col_enc]].copy()
    sem_rota = gdf_plot[~gdf_plot[col_enc]].copy()

    km_max = com_rota[col_km].quantile(0.98) if not com_rota.empty else 1
    km_max = max(km_max, 1)
    km_max = 250
    norm   = mcolors.Normalize(vmin=0, vmax=km_max)
    cmap   = plt.get_cmap(cor_mapa)

    fig, ax = plt.subplots(figsize=(12*centimeters, 12*centimeters), dpi=DPI)
    ax.set_aspect('equal')

    # ── Fundo do estado ──────────────────────────────────────
    if ceara is not None:
        ceara.plot(ax=ax, facecolor="#adadad", edgecolor='#555555', linewidth=0.8, zorder=1)
    else:
        x0, y0, x1, y1 = gdf_plot.total_bounds
        ax.set_xlim(x0 - 0.2, x1 + 0.2)
        ax.set_ylim(y0 - 0.2, y1 + 0.2)
        rect = plt.Rectangle((x0, y0), x1-x0, y1-y0,
                               facecolor="#9E9E9E", edgecolor='#555555', linewidth=0.8)
        ax.add_patch(rect)

    # ── Vias estaduais ────────────────────────────────────────
    n_est = 0
    if gdf_est is not None and not gdf_est.empty:
        gdf_est.plot(ax=ax, color="#e7e6e6", linewidth=0.5, zorder=2, alpha=0.7)
        n_est = len(gdf_est)

    # ── Vias federais ─────────────────────────────────────────
    n_fed = 0
    if gdf_fed is not None and not gdf_fed.empty:
        gdf_fed.plot(ax=ax, color="#45CAFF", linewidth=1.1, zorder=3, alpha=0.9)
        n_fed = len(gdf_fed)

    print(f'  [{analise}] federais={n_fed}  estaduais={n_est}  '
          f'OAEs com rota={len(com_rota)}  sem rota={len(sem_rota)}')

    # ── OAEs com rota — gradiente de cor ──────────────────────
    if not com_rota.empty:
        ax.scatter(
            com_rota.geometry.x, com_rota.geometry.y,
            c=com_rota[col_km], cmap=cmap, norm=norm,
            s=22, zorder=5, edgecolors='white', linewidths=0.3, alpha=0.92,
        )

    # ── OAEs sem rota — X preto ───────────────────────────────
    if not sem_rota.empty:
        ax.scatter(
            sem_rota.geometry.x, sem_rota.geometry.y,
            c=COR_SEM_ROTA, marker='x', s=28, linewidths=1.0, zorder=6,
        )

    # ── Colorbar ──────────────────────────────────────────────
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label('Distância da rota alternativa para a OAE (km)', fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    # ── Legenda ───────────────────────────────────────────────
    handles = []
    if n_fed:
        handles.append(plt.Line2D([0], [0], color='#45CAFF', linewidth=1.5,
                                   label='Via federal (SNV)'))
    if n_est:
        handles.append(plt.Line2D([0], [0], color='#aaaaaa', linewidth=1.0,
                                   label='Via estadual'))
    if not sem_rota.empty:
        handles.append(plt.Line2D([0], [0], marker='x', color=COR_SEM_ROTA,
                                   markersize=6, linewidth=0,
                                   label=f'OAE sem rota alternativa ({len(sem_rota)})'))
    
    handles.append(plt.Line2D(
        [0], [0],
        marker='o', linestyle='',
        markerfacecolor="#ffffff", markeredgecolor='black',
        markersize=7, label='OAE com rota alternativa'
    ))
        
    if handles:
        ax.legend(handles=handles, loc='lower left', fontsize=6, framealpha=0.88)

    # ax.set_title(titulo, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Longitude', fontsize=8)
    ax.set_ylabel('Latitude', fontsize=8)
    ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.savefig(nome_arquivo, dpi=DPI, bbox_inches='tight')
    print(f'  Salvo: {nome_arquivo}')
    plt.show()


_mapa_pontos(
    gdf, 'A2', 'RdYlGn_r',
    titulo='Rotas apenas em vias pavimentadas no SNV',
    nome_arquivo='mapa_rotas_A2.png',
    gdf_fed=rodovias_fed,
    gdf_est=rodovias_est,
)

_mapa_pontos(
    gdf, 'A1', 'RdYlGn_r',
    titulo='Retas em todas as superfícies',
    nome_arquivo='mapa_rotas_A1.png',
    gdf_fed=rodovias_fed,
    gdf_est=rodovias_est,
)

In [ ]:
# Comparação: distância em A1 (sem restrição de superfície) para as OAEs cuja
# Análise 2 (só pavimentado) não encontrou rota, vs. as que tiveram sucesso em
# ambas — embasa a discussão do painel A2 da Figura 1.
falha_a2   = gdf['encontrada_A1'] & ~gdf['encontrada_A2']   # A2 sem rota (mas A1 encontrou)
sucesso_a2 = gdf['encontrada_A1'] & gdf['encontrada_A2']    # ambas encontradas

km_a1_falha_a2   = gdf.loc[falha_a2, 'km_A1'].dropna()
km_a1_sucesso_a2 = gdf.loc[sucesso_a2, 'km_A1'].dropna()

print(f'OAEs sem rota em A2 (mas com rota em A1)  : n={len(km_a1_falha_a2)}')
print(f'  km_A1 média   = {km_a1_falha_a2.mean():.1f} km')
print(f'  km_A1 mediana = {km_a1_falha_a2.median():.1f} km')
print()
print(f'OAEs com rota em ambas as análises         : n={len(km_a1_sucesso_a2)}')
print(f'  km_A1 média   = {km_a1_sucesso_a2.mean():.1f} km')
print(f'  km_A1 mediana = {km_a1_sucesso_a2.median():.1f} km')
print()
print(f'Razão das médias (falha_A2 / sucesso_A2): {km_a1_falha_a2.mean() / km_a1_sucesso_a2.mean():.2f}')

### Discussão — painel A2: ausência de redundância pavimentada

Ao se restringir as rotas alternativas possíveis para apenas aquelas vias que estejam pavimentadas — e, portanto, com maior probabilidade de absorver o tráfego existente na OAE —, diversas regiões da malha revelam-se sem alternativa ou redundância: 55 das 310 OAEs (17,7%) não têm rota alternativa pavimentada dentro do raio de busca. Notavelmente, essas 55 OAEs já apresentavam, na Análise 1, mediana de distância de desvio de 124,8 km — mais que o dobro da mediana das demais (55,2 km) —, indicando que a fragilidade de redundância nessas regiões não é exclusiva da malha pavimentada, mas reflete uma escassez mais geral de conectividade viária. Isso indica que a redundância da malha viária, nessas regiões, apoia-se principalmente em vias não pavimentadas, que podem demandar obras de engenharia — como reforço estrutural e revestimento — para poderem atender ao volume de tráfego a ser desviado, não se configurando como alternativas imediatas à indisponibilidade da OAE.

## Figura 2 — Histograma das distâncias (A1 e A2)

In [ ]:
km_a1 = gdf.loc[gdf['encontrada_A1'], 'km_A1'].dropna()
km_a2 = gdf.loc[gdf['encontrada_A2'], 'km_A2'].dropna()

# Limite comum dos eixos
km_max_hist = max(km_a1.max(), km_a2.max()) * 1.02
bins = np.arange(0, km_max_hist + 20, 20)   # bins de 20 km

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(km_a1, bins=bins, color=COR_A1, alpha=0.65,
        label=f'A1 — todas as superfícies  (n={len(km_a1)})', edgecolor='white', linewidth=0.4)
ax.hist(km_a2, bins=bins, color=COR_A2, alpha=0.65,
        label=f'A2 — PAV + DUP + EOD  (n={len(km_a2)})', edgecolor='white', linewidth=0.4)

# Medianas
for km_serie, cor, label in [
    (km_a1, COR_A1, f'Mediana A1 = {km_a1.median():.1f} km'),
    (km_a2, COR_A2, f'Mediana A2 = {km_a2.median():.1f} km'),
]:
    ax.axvline(km_serie.median(), color=cor, linestyle='--', linewidth=1.4, label=label)

ax.set_xlabel('Comprimento da rota alternativa (km)', fontsize=10)
ax.set_ylabel('Número de OAEs', fontsize=10)
ax.set_title('Distribuição dos comprimentos das rotas alternativas', fontsize=11, fontweight='bold')
ax.legend(fontsize=8.5, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=9)

# Anotação de OAEs sem rota
n_sem_a1 = (gdf['status_A1'] != 'encontrada').sum()
n_sem_a2 = (gdf['status_A2'] != 'encontrada').sum()
ax.text(0.98, 0.96,
        f'Sem rota A1: {n_sem_a1}  |  Sem rota A2: {n_sem_a2}',
        transform=ax.transAxes, ha='right', va='top', fontsize=8,
        color='#444444', style='italic')

plt.tight_layout()
plt.savefig('histograma_rotas.png', dpi=DPI, bbox_inches='tight')
print('Salvo: histograma_rotas.png')
plt.show()

# ── Diagnóstico de forma da distribuição (A1 e A2) ────────────────
# Mesmo diagnóstico aplicado ao indicador de impacto de tráfego (veículos×km
# de desvio): assimetria (skewness) e teste de normalidade Shapiro-Wilk, nos
# dados brutos e no log (log só é calculável para valores > 0). Estatísticas
# descritivas básicas (mínimo, mediana, média, P90, máximo) já estão na
# seção "Estatísticas resumidas", mais abaixo — aqui o foco é a forma da
# distribuição, não os valores centrais em si.
for nome, serie in [('A1', km_a1), ('A2', km_a2)]:
    pos         = serie[serie > 0]
    n_zero      = int((serie == 0).sum())
    skew_raw    = stats.skew(serie)
    skew_log    = stats.skew(np.log(pos))
    shapiro_raw = stats.shapiro(serie)
    shapiro_log = stats.shapiro(np.log(pos))

    print(f'\nForma da distribuição — Comprimento da rota alternativa ({nome}), n={len(serie)}:')
    print(f'  Razão média/mediana                 : {serie.mean() / serie.median():.2f}')
    print(f'  Assimetria (skewness), dados brutos : {skew_raw:.2f}')
    print(f'  Assimetria (skewness), log(dados)   : {skew_log:.2f}  (excluídos {n_zero} valores == 0)')
    print(f'  Shapiro-Wilk, dados brutos : stat={shapiro_raw.statistic:.3f}, p={shapiro_raw.pvalue:.2e}')
    print(f'  Shapiro-Wilk, log(dados)   : stat={shapiro_log.statistic:.3f}, p={shapiro_log.pvalue:.2e}')

### Discussão — forma da distribuição e escolha da mediana (comprimento da rota alternativa)

Assim como o indicador de impacto de tráfego, a distribuição do comprimento da rota alternativa é assimétrica à direita tanto em A1 quanto em A2: a média supera a mediana em 22% na Análise 1 (75,0 km vs. 61,5 km) e em 26% na Análise 2 (86,1 km vs. 68,1 km), com assimetria (*skewness*) positiva de 0,76 e 0,92, respectivamente — visível no histograma como uma cauda à direita, com um número reduzido de OAEs (principalmente em A2) cujo desvio ultrapassa 250 km.

Como no indicador de impacto de tráfego, o log-transform não aproxima essas distribuições de uma normal: o teste de Shapiro-Wilk rejeita a normalidade do log com força igual ou maior à dos dados brutos em ambas as análises, e a assimetria do log inverte de sinal e cresce em magnitude (A1: de +0,76 para −1,55; A2: de +0,92 para −1,22). O padrão multimodal visível no histograma — concentrações distintas em torno de 0–20 km, ~100 km e ~170–220 km, mais pronunciadas em A2 — é mais consistente com a hierarquia da malha viária mapeada do que com um único processo gerador log-normal: OAEs próximas de vias de hierarquia mais alta e mais conectadas produzem desvios curtos, enquanto OAEs em trechos mais isolados exigem desvios substancialmente maiores, sem um contínuo suave entre esses regimes. Por isso, a mediana — e não a média — é reportada como medida de tendência central para o comprimento da rota alternativa em ambas as análises.

## Figura 3 — Scatter A1 × A2

In [ ]:
# Categorias
ambas   = gdf['encontrada_A1'] & gdf['encontrada_A2']
so_a1   = gdf['encontrada_A1'] & ~gdf['encontrada_A2']
so_a2   = ~gdf['encontrada_A1'] & gdf['encontrada_A2']
nenhuma = ~gdf['encontrada_A1'] & ~gdf['encontrada_A2']

fig, ax = plt.subplots(figsize=(6, 6))

# Diagonal de indiferença (y = x)
lim_max = max(
    gdf.loc[ambas, 'km_A1'].max() if ambas.any() else 0,
    gdf.loc[ambas, 'km_A2'].max() if ambas.any() else 0,
) * 1.05
lim_max = max(lim_max, 50)
ax.plot([0, lim_max], [0, lim_max], color='#aaaaaa', linewidth=1,
        linestyle='--', zorder=1, label='A1 = A2')

# Ambas encontradas
if ambas.any():
    ax.scatter(
        gdf.loc[ambas, 'km_A1'], gdf.loc[ambas, 'km_A2'],
        color='#4dac26', s=18, alpha=0.6, zorder=3,
        label=f'Ambas encontradas (n={ambas.sum()})',
    )

# Só A1 encontrada (A2 falhou)
if so_a1.any():
    ax.scatter(
        gdf.loc[so_a1, 'km_A1'],
        [lim_max * 0.98] * so_a1.sum(),
        marker='^', color=COR_A1, s=35, zorder=4,
        label=f'Só A1 encontrada — A2 falhou (n={so_a1.sum()})',
    )

# Só A2 encontrada (A1 falhou)
if so_a2.any():
    ax.scatter(
        [lim_max * 0.98] * so_a2.sum(),
        gdf.loc[so_a2, 'km_A2'],
        marker='>', color=COR_A2, s=35, zorder=4,
        label=f'Só A2 encontrada — A1 falhou (n={so_a2.sum()})',
    )

# Nenhuma
if nenhuma.any():
    ax.scatter(
        [lim_max * 0.98] * nenhuma.sum(),
        [lim_max * 0.98] * nenhuma.sum(),
        marker='x', color=COR_SEM_ROTA, s=40, linewidths=1.1, zorder=4,
        label=f'Nenhuma encontrada (n={nenhuma.sum()})',
    )

# Regiões de interpretação
ax.fill_between([0, lim_max], [0, lim_max], [lim_max, lim_max],
                color=COR_A2, alpha=0.04, zorder=0)
ax.fill_between([0, lim_max], [0, 0], [0, lim_max],
                color=COR_A1, alpha=0.04, zorder=0)
ax.text(lim_max * 0.55, lim_max * 0.18, 'A2 mais curta',
        fontsize=7.5, color=COR_A1, alpha=0.7, style='italic')
ax.text(lim_max * 0.12, lim_max * 0.62, 'A1 mais curta',
        fontsize=7.5, color=COR_A2, alpha=0.7, style='italic')

ax.set_xlim(0, lim_max)
ax.set_ylim(0, lim_max)
ax.set_xlabel('Rota alternativa A1 (km) — todas as superfícies', fontsize=10)
ax.set_ylabel('Rota alternativa A2 (km) — PAV + DUP + EOD', fontsize=10)
ax.set_title('Comparação entre as análises A1 e A2 por OAE', fontsize=11, fontweight='bold')
ax.legend(fontsize=7.5, framealpha=0.9, loc='upper left')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('scatter_a1_a2.png', dpi=DPI, bbox_inches='tight')
print('Salvo: scatter_a1_a2.png')
plt.show()

## KML de investigação — mapa de calor no Google Earth

Gera `mapa_calor_A1.kml` e `mapa_calor_A2.kml` com o mesmo gradiente de cor do mapa estático.
Cada ponto abre um balão com CodPro, SGE, nome, km da rota e vias usadas — útil para investigar casos individuais.

In [ ]:
import simplekml


def _hex_to_kml(hex_color, alpha=210):
    """Converte '#RRGGBB' para o formato KML 'AABBGGRR'."""
    r = int(hex_color[1:3], 16)
    g = int(hex_color[3:5], 16)
    b = int(hex_color[5:7], 16)
    return f'{alpha:02x}{b:02x}{g:02x}{r:02x}'


def _sge_txt(row):
    v = row.get('SGE')
    return '—' if pd.isna(v) else str(int(v))


def gerar_kml_mapa_calor(gdf_plot, analise, nome_arquivo):
    col_km     = f'km_{analise}'
    col_enc    = f'encontrada_{analise}'
    col_status = f'status_{analise}'
    col_nvias  = f'n_vias_{analise}'
    col_ids    = f'ids_vias_{analise}'

    com_rota = gdf_plot[gdf_plot[col_enc]].copy()
    sem_rota = gdf_plot[~gdf_plot[col_enc]].copy()

    # Mesma escala e colormap do mapa estático
    cmap   = cm.get_cmap('RdYlGn_r')
    km_max = com_rota[col_km].quantile(0.98) if not com_rota.empty else 1
    km_max = max(km_max, 1)
    norm   = mcolors.Normalize(vmin=0, vmax=km_max)

    kml = simplekml.Kml(name=f'Rotas Alternativas — {analise}')

    # ── OAEs com rota ──────────────────────────────────────────
    fld_com = kml.newfolder(name=f'Com rota ({len(com_rota)})')

    for codpro, row in com_rota.iterrows():
        km_val = row[col_km]
        rgba   = cmap(norm(km_val))          # (r, g, b, a) floats 0–1
        hex_c  = '#{:02x}{:02x}{:02x}'.format(
            int(rgba[0] * 255), int(rgba[1] * 255), int(rgba[2] * 255)
        )

        pt = fld_com.newpoint(
            name=f'{codpro}',
            coords=[(row.geometry.x, row.geometry.y)],
        )
        pt.style.iconstyle.color = _hex_to_kml(hex_c)
        pt.style.iconstyle.scale = 0.75
        pt.style.iconstyle.icon.href = (
            'http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png'
        )
        pt.style.labelstyle.scale = 0   # oculta rótulo; aparece só ao clicar
        pt.description = (
            f'<b>CodPro:</b> {codpro}<br/>'
            f'<b>SGE:</b> {_sge_txt(row)}<br/>'
            f'<b>Nome:</b> {row.get("Nome", "—")}<br/>'
            f'<b>Análise:</b> {analise}<br/>'
            f'<b>Status:</b> {row[col_status]}<br/>'
            f'<b>Rota alternativa:</b> {km_val:.2f} km<br/>'
            f'<b>Nº de vias:</b> {int(row.get(col_nvias, 0) or 0)}<br/>'
            f'<b>Vias:</b> {row.get(col_ids, "—")}'
        )

    # ── OAEs sem rota ──────────────────────────────────────────
    if not sem_rota.empty:
        fld_sem = kml.newfolder(name=f'Sem rota ({len(sem_rota)})')
        for codpro, row in sem_rota.iterrows():
            pt = fld_sem.newpoint(
                name=f'{codpro} — SEM ROTA',
                coords=[(row.geometry.x, row.geometry.y)],
            )
            pt.style.iconstyle.color = 'ff0000ff'   # vermelho opaco (KML: AABBGGRR)
            pt.style.iconstyle.scale = 0.9
            pt.style.iconstyle.icon.href = (
                'http://maps.google.com/mapfiles/kml/shapes/cross-hairs.png'
            )
            pt.style.labelstyle.scale = 0
            pt.description = (
                f'<b>CodPro:</b> {codpro}<br/>'
                f'<b>SGE:</b> {_sge_txt(row)}<br/>'
                f'<b>Nome:</b> {row.get("Nome", "—")}<br/>'
                f'<b>Análise:</b> {analise}<br/>'
                f'<b>Status:</b> {row[col_status]}'
            )

    kml.save(nome_arquivo)
    print(f'Salvo: {nome_arquivo}  ({len(gdf_plot)} pontos, escala 0–{km_max:.0f} km)')


gerar_kml_mapa_calor(gdf, 'A1', 'mapa_calor_A1.kml')
gerar_kml_mapa_calor(gdf, 'A2', 'mapa_calor_A2.kml')

## Estatísticas resumidas

In [ ]:
total = len(gdf)
print(f'Total de OAEs: {total}')
print()

for label, col_km, col_enc in [('A1', 'km_A1', 'encontrada_A1'), ('A2', 'km_A2', 'encontrada_A2')]:
    km = gdf.loc[gdf[col_enc], col_km].dropna()
    n_enc = gdf[col_enc].sum()
    print(f'Análise {label}:')
    print(f'  Rotas encontradas : {n_enc} ({n_enc/total*100:.1f}%)')
    print(f'  Sem rota          : {total - n_enc} ({(total - n_enc)/total*100:.1f}%)')
    if not km.empty:
        print(f'  Mínimo   : {km.min():.1f} km')
        print(f'  Mediana  : {km.median():.1f} km')
        print(f'  Média    : {km.mean():.1f} km')
        print(f'  P90      : {km.quantile(0.90):.1f} km')
        print(f'  Máximo   : {km.max():.1f} km')
    print()

# VMD vs. rota alternativa

In [ ]:
# Carrega a planilha de controle geral com VMD
df_vmd = pd.read_excel('controle_geral.xlsx', sheet_name='Planilha1')
df_vmd = df_vmd.set_index('CodPro')

# Faz o join com o GeoDataFrame existente (chave: CodPro)
gdf = gdf.join(df_vmd[['VMDa']], how='left')

print(f'VMDa adicionado ao GeoDataFrame')
print(gdf[['Nome', 'km_A1', 'km_A2', 'VMDa']].head(10))

In [ ]:
# Calculate "Veículos por km de desvio"
gdf['veiculo_km_desvio'] = (gdf['km_A2'] * gdf['VMDa'])

# Filter only rows where A2 was found and VMDa is not null
veiculo_km = gdf.loc[gdf['encontrada_A2'] & gdf['VMDa'].notna(), 'veiculo_km_desvio'].dropna()

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(veiculo_km, bins=30, color=COR_A2, alpha=0.7,
    label=f'Veículos por km de desvio (n={len(veiculo_km)})', edgecolor='white', linewidth=0.4)

ax.axvline(veiculo_km.median(), color=COR_A2, linestyle='--', linewidth=1.4, 
       label=f'Mediana = {veiculo_km.median():.0f} veículos·km')

ax.set_xlabel('Veículos por km de desvio', fontsize=10)
ax.set_ylabel('Número de OAEs', fontsize=10)
ax.set_title('Distribuição do impacto de tráfego nas rotas alternativas A2', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('histograma_veiculo_km_desvio.png', dpi=DPI, bbox_inches='tight')
print('Salvo: histograma_veiculo_km_desvio.png')
plt.show()

# ── Diagnóstico de forma da distribuição ──────────────────────────
# Assimetria (skewness) e teste de normalidade Shapiro-Wilk, nos dados
# brutos e no log dos dados (log só é calculável para valores > 0; os
# valores == 0 ocorrem quando o VMDa cadastrado da via é nulo).
_pos          = veiculo_km[veiculo_km > 0]
_n_zero       = int((veiculo_km == 0).sum())
_skew_raw     = stats.skew(veiculo_km)
_skew_log     = stats.skew(np.log(_pos))
_shapiro_raw  = stats.shapiro(veiculo_km)
_shapiro_log  = stats.shapiro(np.log(_pos))

print(f'\nEstatísticas — Veículos por km de desvio (A2):')
print(f'  Mínimo   : {veiculo_km.min():.0f}')
print(f'  Mediana  : {veiculo_km.median():.0f}')
print(f'  Média    : {veiculo_km.mean():.0f}')
print(f'  P90      : {veiculo_km.quantile(0.90):.0f}')
print(f'  Máximo   : {veiculo_km.max():.0f}')
print(f'  Razão média/mediana                 : {veiculo_km.mean() / veiculo_km.median():.2f}')
print(f'  Assimetria (skewness), dados brutos : {_skew_raw:.2f}')
print(f'  Assimetria (skewness), log(dados)   : {_skew_log:.2f}  (excluídos {_n_zero} valores == 0)')
print(f'  Shapiro-Wilk, dados brutos : stat={_shapiro_raw.statistic:.3f}, p={_shapiro_raw.pvalue:.2e}')
print(f'  Shapiro-Wilk, log(dados)   : stat={_shapiro_log.statistic:.3f}, p={_shapiro_log.pvalue:.2e}')

### Discussão — forma da distribuição e escolha da mediana

A distribuição do indicador veículos×km de desvio é fortemente assimétrica à direita: a média (276.221 veículos·km) supera a mediana (241.954 veículos·km) em cerca de 14%, com assimetria (*skewness*) positiva de 0,77 nos dados brutos — condizente com a cauda longa à direita visível no histograma, formada por um pequeno número de OAEs que combinam alto volume de tráfego (VMDa) com desvios longos. Isso já é suficiente para justificar o uso da mediana, e não da média, como estatística-resumo: a média é puxada por esses poucos casos extremos e não representa bem o "caso típico" do universo estudado.

Vale qualificar, porém, a hipótese de que a distribuição seja log-normal. Aplicando o teste de Shapiro-Wilk sobre o logaritmo dos valores (excluídos os 5 casos com veículos×km = 0, onde o VMDa cadastrado da via é nulo), a hipótese de normalidade do log é rejeitada com *ainda mais força* (p ≈ 2×10⁻¹³) do que a dos dados brutos (p ≈ 4×10⁻¹¹) — ou seja, log-transformar os dados não os aproxima de uma normal; pelo contrário, a assimetria do log (≈ −1,4) inverte de sinal e cresce em magnitude relativa à dos dados brutos (+0,77). O padrão do histograma — concentração de valores baixos, um vale entre ~200 e ~380 mil veículos·km, e um segundo agrupamento entre ~500 e ~900 mil — sugere uma distribuição multimodal, mais provavelmente refletindo a combinação de duas variáveis com escalas próprias (o VMDa, concentrado em faixas típicas de tráfego das rodovias cadastradas, e a extensão do desvio A2) do que um único processo gerador log-normal. Ainda assim, a assimetria à direita é real e robusta o suficiente para manter a mediana como a medida de tendência central reportada para este indicador.

In [ ]:
import matplotlib.ticker as mticker

def _mapa_pontos_vmd(gdf_plot, analise, cor_mapa, titulo, nome_arquivo,
                     gdf_fed=None, gdf_est=None):
    col_km  = f'km_{analise}'
    col_enc = f'encontrada_{analise}'
    col_vmd_impact = 'veiculo_km_desvio'

    com_rota = gdf_plot[gdf_plot[col_enc] & gdf_plot[col_vmd_impact].notna()].copy()
    sem_rota = gdf_plot[~gdf_plot[col_enc]].copy()

    vmd_max = com_rota[col_vmd_impact].quantile(0.98) if not com_rota.empty else 1
    vmd_max = max(vmd_max, 1)
    norm   = mcolors.Normalize(vmin=0, vmax=vmd_max)
    cmap   = plt.get_cmap(cor_mapa)

    fig, ax = plt.subplots(figsize=(12*centimeters, 12*centimeters), dpi=DPI)
    ax.set_aspect('equal')

    # ── Fundo do estado ──────────────────────────────────────
    if ceara is not None:
        ceara.plot(ax=ax, facecolor="#adadad", edgecolor='#555555', linewidth=0.8, zorder=1)
    else:
        x0, y0, x1, y1 = gdf_plot.total_bounds
        ax.set_xlim(x0 - 0.2, x1 + 0.2)
        ax.set_ylim(y0 - 0.2, y1 + 0.2)
        rect = plt.Rectangle((x0, y0), x1-x0, y1-y0,
                               facecolor="#9E9E9E", edgecolor='#555555', linewidth=0.8)
        ax.add_patch(rect)

    # ── Vias estaduais ────────────────────────────────────────
    n_est = 0
    if gdf_est is not None and not gdf_est.empty:
        gdf_est.plot(ax=ax, color="#e7e6e6", linewidth=0.5, zorder=2, alpha=0.7)
        n_est = len(gdf_est)

    # ── Vias federais ─────────────────────────────────────────
    n_fed = 0
    if gdf_fed is not None and not gdf_fed.empty:
        gdf_fed.plot(ax=ax, color="#45CAFF", linewidth=1.1, zorder=3, alpha=0.9)
        n_fed = len(gdf_fed)

    print(f'  [{analise}] federais={n_fed}  estaduais={n_est}  '
          f'OAEs com rota={len(com_rota)}  sem rota={len(sem_rota)}')

    # ── OAEs com rota — gradiente de cor por impacto de tráfego ──────────
    if not com_rota.empty:
        ax.scatter(
            com_rota.geometry.x, com_rota.geometry.y,
            c=com_rota[col_vmd_impact], cmap=cmap, norm=norm,
            s=22, zorder=5, edgecolors='white', linewidths=0.3, alpha=0.92,
        )

    # ── OAEs sem rota — X preto ───────────────────────────────
    if not sem_rota.empty:
        ax.scatter(
            sem_rota.geometry.x, sem_rota.geometry.y,
            c=COR_SEM_ROTA, marker='x', s=28, linewidths=1.0, zorder=6,
        )

    # ── Colorbar ──────────────────────────────────────────────
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.075)

    formatter = mticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 0))  # forces scientific notation
    cbar.formatter = formatter
    cbar.update_ticks()

    cbar.set_label('Impacto de tráfego: km de rota × VMDa (veículos·km)', fontsize=9)
    cbar.ax.tick_params(labelsize=8)
    
    # Shrink the offset text (10^5) by 80%
    cbar.ax.yaxis.get_offset_text().set_fontsize(8)

    # ── Legenda ───────────────────────────────────────────────
    handles = []
    if n_fed:
        handles.append(plt.Line2D([0], [0], color='#45CAFF', linewidth=1.5,
                                   label='Via federal (SNV)'))
    if n_est:
        handles.append(plt.Line2D([0], [0], color='#aaaaaa', linewidth=1.0,
                                   label='Via estadual'))
    if not sem_rota.empty:
        handles.append(plt.Line2D([0], [0], marker='x', color=COR_SEM_ROTA,
                                   markersize=6, linewidth=0,
                                   label=f'OAE sem rota alternativa ({len(sem_rota)})'))
    
    handles.append(plt.Line2D(
        [0], [0],
        marker='o', linestyle='',
        markerfacecolor="#ffffff", markeredgecolor='black',
        markersize=7, label='OAE com rota alternativa'
    ))
        
    if handles:
        ax.legend(handles=handles, loc='lower left', fontsize=6, framealpha=0.88)

    # ax.set_title(titulo, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Longitude', fontsize=8)
    ax.set_ylabel('Latitude', fontsize=8)
    ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.savefig(nome_arquivo, dpi=DPI, bbox_inches='tight')
    print(f'  Salvo: {nome_arquivo}')
    plt.show()


_mapa_pontos_vmd(
    gdf, 'A2', 'RdYlGn_r',
    titulo='Impacto de tráfego nas rotas alternativas (km × VMDa)',
    nome_arquivo='mapa_impacto_veiculo_km_A2.png',
    gdf_fed=rodovias_fed,
    gdf_est=rodovias_est,
)

In [ ]:
# Correlação entre extensão do desvio (km_A2) e VMDa — embasa a discussão de
# que OAEs com desvios mais longos não são, em geral, as de maior impacto:
# elas tendem a estar em corredores de tráfego mais baixo.
mask_impacto = gdf['encontrada_A2'] & gdf['VMDa'].notna()
sub = gdf.loc[mask_impacto, ['Nome', 'km_A2', 'VMDa', 'veiculo_km_desvio']].copy()

r_pearson,  p_pearson  = stats.pearsonr(sub['km_A2'], sub['VMDa'])
r_spearman, p_spearman = stats.spearmanr(sub['km_A2'], sub['VMDa'])
r_imp,      p_imp      = stats.spearmanr(sub['km_A2'], sub['veiculo_km_desvio'])

print(f'Correlação entre extensão do desvio (km_A2) e VMDa da via interditada (n={len(sub)}):')
print(f'  Pearson  : r   = {r_pearson:.3f}   p = {p_pearson:.2e}')
print(f'  Spearman : rho = {r_spearman:.3f}   p = {p_spearman:.2e}')
print()
print(f'Correlação entre extensão do desvio (km_A2) e o indicador de impacto (veículos×km):')
print(f'  Spearman : rho = {r_imp:.3f}   p = {p_imp:.2e}')

# Exemplos concretos: OAEs com os maiores desvios x sua posição no ranking de impacto
top_km = sub.nlargest(10, 'km_A2').copy()
top_km['rank_impacto'] = sub['veiculo_km_desvio'].rank(ascending=False).loc[top_km.index].astype(int)
print(f'\nTop 10 OAEs por extensão do desvio (km_A2) — posição delas no ranking de impacto (1 = maior impacto, de {len(sub)}):')
print(top_km[['Nome', 'km_A2', 'VMDa', 'rank_impacto']].to_string(index=True))

# OAEs no topo do ranking de impacto, para contraste
top_impacto = sub.nlargest(10, 'veiculo_km_desvio')
print(f'\nTop 10 OAEs por indicador de impacto (veículos×km de desvio):')
print(top_impacto[['Nome', 'km_A2', 'VMDa', 'veiculo_km_desvio']].to_string(index=True))

### Discussão — impacto de tráfego vs. extensão do desvio

O reordenamento visível entre o mapa de distância (Figura 1, painel A2) e o mapa de impacto acima é substancial, e não por acaso: a correlação entre a extensão do desvio e o VMDa da via interditada é **negativa** (Pearson r = −0,30). OAEs em corredores rodoviários isolados apresentam os maiores desvios absolutos, mas justamente por estarem nesses corredores tendem a atender volumes de tráfego relativamente baixos — o que as desloca para posições intermediárias no ranking de impacto, apesar de aparecerem em vermelho no mapa de distância pura. Em contraste, OAEs com desvios moderados situadas em corredores federais de tráfego intenso assumem as posições mais críticas do indicador composto. Ainda assim, a correlação entre extensão do desvio e o indicador de impacto permanece positiva e forte (Spearman rho = 0,76): a distância continua sendo um componente relevante, mas a ponderação pelo VMDa reordena de forma não trivial quais OAEs merecem prioridade — um resultado que a análise de redundância física, isoladamente, não capturaria.

In [ ]:
import matplotlib.ticker as mticker

def _mapa_pontos_vmd(gdf_plot, analise, cor_mapa, titulo, nome_arquivo,
                     gdf_fed=None, gdf_est=None):
    col_km  = f'km_{analise}'
    col_enc = f'encontrada_{analise}'
    col_vmd_impact = 'veiculo_km_desvio'

    com_rota = gdf_plot[gdf_plot[col_enc] & gdf_plot[col_vmd_impact].notna()].copy()
    sem_rota = gdf_plot[~gdf_plot[col_enc]].copy()

    vmd_max = com_rota[col_vmd_impact].quantile(0.98) if not com_rota.empty else 1
    vmd_max = max(vmd_max, 1)
    norm   = mcolors.Normalize(vmin=0, vmax=vmd_max)
    cmap   = plt.get_cmap(cor_mapa)

    fig, ax = plt.subplots(figsize=(12*centimeters, 12*centimeters), dpi=DPI)
    ax.set_aspect('equal')

    # ── Fundo do estado ──────────────────────────────────────
    if ceara is not None:
        ceara.plot(ax=ax, facecolor="#adadad", edgecolor='#555555', linewidth=0.8, zorder=1)
    else:
        x0, y0, x1, y1 = gdf_plot.total_bounds
        ax.set_xlim(x0 - 0.2, x1 + 0.2)
        ax.set_ylim(y0 - 0.2, y1 + 0.2)
        rect = plt.Rectangle((x0, y0), x1-x0, y1-y0,
                               facecolor="#9E9E9E", edgecolor='#555555', linewidth=0.8)
        ax.add_patch(rect)

    # ── Vias estaduais ────────────────────────────────────────
    n_est = 0
    if gdf_est is not None and not gdf_est.empty:
        gdf_est.plot(ax=ax, color="#e7e6e6", linewidth=0.5, zorder=2, alpha=0.7)
        n_est = len(gdf_est)

    # ── Vias federais ─────────────────────────────────────────
    n_fed = 0
    if gdf_fed is not None and not gdf_fed.empty:
        gdf_fed.plot(ax=ax, color="#45CAFF", linewidth=1.1, zorder=3, alpha=0.9)
        n_fed = len(gdf_fed)

    print(f'  [{analise}] federais={n_fed}  estaduais={n_est}  '
          f'OAEs com rota={len(com_rota)}  sem rota={len(sem_rota)}')

    # ── OAEs com rota — gradiente de cor por impacto de tráfego ──────────
    if not com_rota.empty:
        ax.scatter(
            com_rota.geometry.x, com_rota.geometry.y,
            c=com_rota[col_vmd_impact], cmap=cmap, norm=norm,
            s=22, zorder=5, edgecolors='white', linewidths=0.3, alpha=0.92,
        )

    # ── OAEs sem rota — X preto ───────────────────────────────
    if not sem_rota.empty:
        ax.scatter(
            sem_rota.geometry.x, sem_rota.geometry.y,
            c=COR_SEM_ROTA, marker='x', s=28, linewidths=1.0, zorder=6,
        )

    # ── Colorbar ──────────────────────────────────────────────
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.075)

    formatter = mticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 0))  # forces scientific notation
    cbar.formatter = formatter
    cbar.update_ticks()

    cbar.set_label('Impacto de tráfego: km de rota × VMDa (veículos·km)', fontsize=9)
    cbar.ax.tick_params(labelsize=8)
    
    # Shrink the offset text (10^5) by 80%
    cbar.ax.yaxis.get_offset_text().set_fontsize(8)

    # ── Legenda ───────────────────────────────────────────────
    handles = []
    if n_fed:
        handles.append(plt.Line2D([0], [0], color='#45CAFF', linewidth=1.5,
                                   label='Via federal (SNV)'))
    if n_est:
        handles.append(plt.Line2D([0], [0], color='#aaaaaa', linewidth=1.0,
                                   label='Via estadual'))
    if not sem_rota.empty:
        handles.append(plt.Line2D([0], [0], marker='x', color=COR_SEM_ROTA,
                                   markersize=6, linewidth=0,
                                   label=f'OAE sem rota alternativa ({len(sem_rota)})'))
    
    handles.append(plt.Line2D(
        [0], [0],
        marker='o', linestyle='',
        markerfacecolor="#ffffff", markeredgecolor='black',
        markersize=7, label='OAE com rota alternativa'
    ))
        
    if handles:
        ax.legend(handles=handles, loc='lower left', fontsize=6, framealpha=0.88)

    # ax.set_title(titulo, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Longitude', fontsize=8)
    ax.set_ylabel('Latitude', fontsize=8)
    ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.savefig(nome_arquivo, dpi=DPI, bbox_inches='tight')
    print(f'  Salvo: {nome_arquivo}')
    plt.show()


_mapa_pontos_vmd(
    gdf, 'A1', 'RdYlGn_r',
    titulo='Impacto de tráfego nas rotas alternativas (km × VMDa)',
    nome_arquivo='mapa_impacto_veiculo_km_A1.png',
    gdf_fed=rodovias_fed,
    gdf_est=rodovias_est,
)